In [7]:
# %load_ext cudf.pandas
import numpy as np
import pandas as pd
import cudf as cd

## modifying

In [2]:
a = np.random.randint(0, 100, (100000, 3))
df = pd.DataFrame(a)

In [3]:
# move to gpu
df_col0 = cd.Series.from_pandas(df[0])
print(type(df_col0))

<class 'cudf.core.series.Series'>


In [4]:
%%cudf.pandas.profile
df_col0 = df[0] + 1

                                                                                                           
                                         Total time elapsed: 0.083 seconds                                 
                                       2 GPU function calls in 0.013 seconds                               
                                       0 CPU function calls in 0.000 seconds                               
                                                                                                           
                                                       Stats                                               
                                                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function              ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__ │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__add__      │ 1          │ 0.010       │ 0.010       │ 0          │ 0.000       │ 0.000       │
└───────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Generating '/var/tmp/nsys-report-cbc6.qdstrm'
[1/2] [========================100%] test_2025-04-28T06:46:33.nsys-rep
[2/2] [========================100%] test_2025-04-28T06:46:33.sqlite
Generated:
    /home/dias-benchmarks/cost-modelling/test_2025-04-28T06:46:33.nsys-rep
    /home/dias-benchmarks/cost-modelling/test_2025-04-28T06:46:33.sqlite
Generating SQLite file /home/dias-benchmarks/cost-modelling/test_2025-04-28T06:46:33.sqlite from /home/dias-benchmarks/cost-modelling/test_2025-04-28T06:46:33.nsys-rep
Processing [/home/dias-benchmarks/cost-modelling/test_2025-04-28T06:46:33.sqlite] with [/opt/nvidia/nsight-systems/2024.1.1/host-linux-x64/reports/nvtx_sum.py]... 

 ** NVTX Range Summary (nvtx_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)    Med (ns)   Min (ns)  Max (ns)  StdDev (ns)   Style                           Range                         
 --------  ---------------  ---------  ----------  ----------  --------  --------  -----------  -------  ---------------------

In [10]:
%%cudf.pandas.profile
df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   0       100000 non-null  int64
 1   1       100000 non-null  int64
 2   2       100000 non-null  int64
dtypes: int64(3)
memory usage: 2.3 MB


                                                                                                    
                                     Total time elapsed: 0.094 seconds                              
                                   1 GPU function calls in 0.024 seconds                            
                                   0 CPU function calls in 0.000 seconds                            
                                                                                                    
                                                   Stats                                            
                                                                                                    
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function       ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.info │ 1          │ 0.024       │ 0.024       │ 0          │ 0.000       │ 0.000       │
└────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [ ]:
# move to cpu
df_col0 = df_col0.to_pandas()
print(type(df_col0))

In [ ]:
df[0] = df_col0

## inserting a new column

In [ ]:
# move to gpu
df_cudf = cd.DataFrame.from_pandas(df[[0, 1]])

In [ ]:
df_cudf[3] = df_cudf[0] + df_cudf[1]
df_cudf.columns

In [ ]:
# move to cpu
df_cudf = df_cudf.to_pandas()

In [ ]:
diff_gpu_to_cpu = list(set(df_cudf.columns) - set(df.columns))

In [ ]:
df[diff_gpu_to_cpu] = df_cudf[diff_gpu_to_cpu]

## deleting a new column

In [ ]:
# move to gpu
df_cudf = cd.DataFrame.from_pandas(df[[0, 1]])

In [ ]:
df_cudf = df_cudf.drop(0, axis=1)
df_cudf.columns

In [ ]:
diff_gpu_to_cpu = list(set([0, 1]) - set(df_cudf.columns))

In [ ]:
df = df.drop(diff_gpu_to_cpu, axis=1)

## testing small data


In [22]:
small_data = np.random.randint(0, 100, (10000, 1)).astype(str)
large_data = np.random.randint(0, 100, (1000000, 10)).astype(str)

In [23]:
small_df = pd.DataFrame(small_data)

In [24]:
large_df = pd.DataFrame(large_data)

In [13]:
small_df = cd.DataFrame.from_pandas(small_df)
large_df = cd.DataFrame.from_pandas(large_df)

In [19]:
%%timeit
small_df[0].drop_duplicates()

887 µs ± 2.58 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [21]:
%%timeit
large_df[0].drop_duplicates()

2.24 ms ± 22.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
